# LeetCode #1066: Campus Bikes II

https://leetcode.com/problems/campus-bikes-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(m! / (m-n)!)$ | $O(n)$ |
| **Optimal: Bitmask DP ★** | $O(2^m \times m)$ | $O(2^m)$ |

---

## Understanding the Methods

### Brute Force
Try all permutations of bike assignments to workers. With $n$ workers and $m$ bikes, this is $P(m,n)$ permutations — factorial complexity that becomes infeasible quickly.

### Optimal: Bitmask DP ★
Use a bitmask to represent which bikes have been assigned. `dp[mask]` stores the minimum total Manhattan distance when the bikes in `mask` have been assigned to the first `popcount(mask)` workers in order. For each state, try assigning each unassigned bike to the next worker. The number of states is $2^m$ and each has at most $m$ transitions.

**Constraints:**
* $1 \leq n \leq m \leq 10$
* `workers[i]`, `bikes[i]` are coordinates in $[0, 999]$

## Solutions

### C#

In [ ]:
public class Solution {
    public int AssignBikes(int[][] workers, int[][] bikes) {
        int n = workers.Length, m = bikes.Length;
        int[] dp = new int[1 << m];
        Array.Fill(dp, int.MaxValue);
        dp[0] = 0;
        int ans = int.MaxValue;
        for (int mask = 0; mask < (1 << m); mask++) {
            if (dp[mask] == int.MaxValue) continue;
            // The number of set bits tells us which worker is next to assign
            int w = BitCount(mask);
            if (w == n) { ans = Math.Min(ans, dp[mask]); continue; }
            for (int b = 0; b < m; b++) {
                if ((mask >> b & 1) == 1) continue; // bike already taken
                int next = mask | (1 << b);
                // Assign bike b to worker w and propagate the minimum cost
                int dist = Math.Abs(workers[w][0] - bikes[b][0]) + Math.Abs(workers[w][1] - bikes[b][1]);
                dp[next] = Math.Min(dp[next], dp[mask] + dist);
            }
        }
        return ans;
    }

    private int BitCount(int x) {
        int c = 0; while (x != 0) { c += x & 1; x >>= 1; } return c;
    }
}

### Python

In [ ]:
class Solution:
    def assign_bikes(self, workers: list[list[int]], bikes: list[list[int]]) -> int:
        n, m = len(workers), len(bikes)
        dp = [float('inf')] * (1 << m)
        dp[0] = 0
        ans = float('inf')
        for mask in range(1 << m):
            if dp[mask] == float('inf'): continue
            # The number of set bits tells us which worker is next to assign
            w = bin(mask).count('1')
            if w == n: ans = min(ans, dp[mask]); continue
            for b in range(m):
                if mask >> b & 1: continue  # bike already taken
                nxt = mask | (1 << b)
                # Assign bike b to worker w and propagate the minimum cost
                dist = abs(workers[w][0] - bikes[b][0]) + abs(workers[w][1] - bikes[b][1])
                dp[nxt] = min(dp[nxt], dp[mask] + dist)
        return ans

### Go

In [ ]:
import "math/bits"

func assignBikes(workers [][]int, bikes [][]int) int {
    n, m := len(workers), len(bikes)
    dp := make([]int, 1<<m)
    for i := range dp { dp[i] = 1<<31 - 1 }
    dp[0] = 0
    ans := 1<<31 - 1
    for mask := 0; mask < (1 << m); mask++ {
        if dp[mask] == 1<<31-1 { continue }
        // The number of set bits tells us which worker is next to assign
        w := bits.OnesCount(uint(mask))
        if w == n { if dp[mask] < ans { ans = dp[mask] }; continue }
        for b := 0; b < m; b++ {
            if mask>>b&1 == 1 { continue } // bike already taken
            nxt := mask | (1 << b)
            // Assign bike b to worker w and propagate the minimum cost
            d := abs1066(workers[w][0]-bikes[b][0]) + abs1066(workers[w][1]-bikes[b][1])
            if dp[mask]+d < dp[nxt] { dp[nxt] = dp[mask] + d }
        }
    }
    return ans
}
func abs1066(x int) int { if x < 0 { return -x }; return x }

### Rust

In [ ]:
impl Solution {
    pub fn assign_bikes(workers: Vec<Vec<i32>>, bikes: Vec<Vec<i32>>) -> i32 {
        let (n, m) = (workers.len(), bikes.len());
        let mut dp = vec![i32::MAX; 1 << m];
        dp[0] = 0;
        let mut ans = i32::MAX;
        for mask in 0..(1usize << m) {
            if dp[mask] == i32::MAX { continue; }
            // The number of set bits tells us which worker is next to assign
            let w = mask.count_ones() as usize;
            if w == n { ans = ans.min(dp[mask]); continue; }
            for b in 0..m {
                if (mask >> b) & 1 == 1 { continue; } // bike already taken
                let nxt = mask | (1 << b);
                // Assign bike b to worker w and propagate the minimum cost
                let dist = (workers[w][0] - bikes[b][0]).abs() + (workers[w][1] - bikes[b][1]).abs();
                if dp[mask] != i32::MAX {
                    dp[nxt] = dp[nxt].min(dp[mask] + dist);
                }
            }
        }
        ans
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `workers = [[0,0],[2,1]]`, `bikes = [[1,2],[3,3]]`
Assignment (worker 0 → bike 0, worker 1 → bike 1): cost $= 1+2+1+2 = 6$. Assignment (worker 0 → bike 1, worker 1 → bike 0): cost $= 3+3+2+1 = 9$. The bitmask DP picks $6$.

### 2. Slightly Complex
**Input:** `workers = [[0,0],[1,1],[2,0]]`, `bikes = [[1,0],[2,2],[2,1]]`
Three workers, three bikes: $3! = 6$ assignments tried. The DP evaluates all via the $2^3 = 8$ bitmask states and finds the optimal pairing.

### 3. Edge Case: Time Factor
**Input:** `n = 10 workers`, `m = 10 bikes`, random positions
Maximum input: $2^{10} = 1024$ states, each trying up to $10$ transitions $= 10{,}240$ operations. Feasible even with the full exponent.

### 4. Edge Case: Space Factor
**Input:** `n = 1`, `m = 1`
Single worker and single bike; the DP table has $2$ entries. Only one assignment is possible — returns the single Manhattan distance.

### 5. Almost-Impossible but Plausible
**Input:** All workers at $(0,0)$, all bikes at $(999,999)$, `n = m = 10`
Every assignment has the same total distance $= n \times (999 + 999) = 19{,}980$. The DP still evaluates all paths but every `dp[next]` converges to the same value — confirming the algorithm handles uniform cost landscapes without false optima.